# Random Forest Model

In [ ]:
import numpy as np
import pandas as pd
import joblib
from matplotlib import pyplot as plt
from sklearn import metrics
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import (
    train_test_split, KFold, cross_validate, GridSearchCV,
)
from sklearn.preprocessing import OneHotEncoder

## 1. Load & Clean Data

In [ ]:
train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")
test_ids = test_df["id"].copy()

train_df = train_df.drop(columns=["id"])
test_df = test_df.drop(columns=["id"])

train_df["internet_access"] = train_df["internet_access"].map({"yes": 1, "no": 0})
test_df["internet_access"] = test_df["internet_access"].map({"yes": 1, "no": 0})

categorical_features = [
    "gender", "course", "sleep_quality",
    "study_method", "facility_rating", "exam_difficulty",
]
encoder = OneHotEncoder(drop="first", sparse_output=False)
encoded_train = encoder.fit_transform(train_df[categorical_features])
encoded_test = encoder.transform(test_df[categorical_features])
encoded_names = encoder.get_feature_names_out(categorical_features)

train_df = pd.concat(
    [train_df.drop(columns=categorical_features),
     pd.DataFrame(encoded_train, columns=encoded_names)],
    axis=1,
)
test_df = pd.concat(
    [test_df.drop(columns=categorical_features),
     pd.DataFrame(encoded_test, columns=encoded_names)],
    axis=1,
)

target = "exam_score"
y = train_df[target]
X = train_df.drop(columns=[target])
X_submission = test_df[X.columns]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42,
)

print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_submission.shape}")
X_train.head()

## 2. EDA & Baseline

Train a default `RandomForestRegressor` on the training split to establish a reference score.

In [ ]:
baseline_rf = RandomForestRegressor(random_state=42, n_jobs=-1)
baseline_rf.fit(X_train, y_train)

y_baseline_pred = baseline_rf.predict(X_val)

baseline_metrics = pd.DataFrame({
    "MAE":  [metrics.mean_absolute_error(y_val, y_baseline_pred)],
    "MSE":  [metrics.mean_squared_error(y_val, y_baseline_pred)],
    "RMSE": [np.sqrt(metrics.mean_squared_error(y_val, y_baseline_pred))],
    "R2":   [metrics.r2_score(y_val, y_baseline_pred)],
})

print("Baseline (default RF) — hold-out validation:")
baseline_metrics

## 3. Define Validation Strategy (5-Fold CV)

Use 5-fold cross-validation on the training set to get a more robust estimate of baseline performance.

Pattern adapted from `initial_linear_regression.ipynb`.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "MAE":  "neg_mean_absolute_error",
    "MSE":  "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2":   "r2",
}

cv_scores = cross_validate(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    X_train, y_train,
    cv=cv,
    scoring=scoring,
)

cv_baseline = pd.DataFrame({
    "MAE":  [-cv_scores["test_MAE"].mean()],
    "MSE":  [-cv_scores["test_MSE"].mean()],
    "RMSE": [-cv_scores["test_RMSE"].mean()],
    "R2":   [cv_scores["test_R2"].mean()],
})

print("Baseline (default RF) — 5-Fold CV on training set:")
cv_baseline

## 4. Hyperparameter Tuning (GridSearchCV + 5-Fold CV)

Search over key Random Forest hyperparameters using `GridSearchCV` with the same
5-fold strategy.  A representative subsample is used to keep runtime manageable
(pattern from `nearest_neighbour.ipynb`).

In [ ]:
SAMPLE_SIZE_TUNE = 50_000

sample_idx = X_train.sample(
    n=min(SAMPLE_SIZE_TUNE, len(X_train)), random_state=42,
).index
X_tune = X_train.loc[sample_idx]
y_tune = y_train.loc[sample_idx]
print(f"Tuning on {len(X_tune):,} samples")

In [ ]:
param_grid = {
    "n_estimators":    [100, 200, 300],
    "max_depth":       [None, 15, 25],
    "min_samples_leaf": [1, 2, 4],
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)
grid.fit(X_tune, y_tune)

print(f"\nBest params: {grid.best_params_}")
print(f"Best CV RMSE (tune sample): {-grid.best_score_:.4f}")

cv_results = pd.DataFrame(grid.cv_results_).sort_values("rank_test_score")
cv_results["RMSE"] = -cv_results["mean_test_score"]
cv_results[
    ["param_n_estimators", "param_max_depth", "param_min_samples_leaf", "RMSE"]
].head(10)

## 5. Final Training with Best Hyperparameters

Retrain on the **full training split** using the best hyperparameters found above.

In [ ]:
best_params = grid.best_params_
print(f"Training final model with: {best_params}")

rf = RandomForestRegressor(
    **best_params,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
print(f"Fitted on {len(X_train):,} training samples")

## 6. Final Evaluation

Evaluate on the held-out validation set and compare against the baseline.

In [ ]:
y_val_pred = rf.predict(X_val)

plt.scatter(y_val, y_val_pred, alpha=0.4)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], "r", lw=2)
plt.xlabel("Actual Exam Score")
plt.ylabel("Predicted Exam Score")
plt.title("Validation: Actual vs Predicted (Tuned RF)")
plt.show()

plt.hist(y_val - y_val_pred, bins=50)
plt.xlabel("Prediction Error")
plt.title("Validation: Error Distribution (Tuned RF)")
plt.show()

In [ ]:
pd.set_option("float_format", "{:f}".format)

tuned_metrics = pd.DataFrame({
    "MAE":  [metrics.mean_absolute_error(y_val, y_val_pred)],
    "MSE":  [metrics.mean_squared_error(y_val, y_val_pred)],
    "RMSE": [np.sqrt(metrics.mean_squared_error(y_val, y_val_pred))],
    "R2":   [metrics.r2_score(y_val, y_val_pred)],
})

comparison = pd.concat(
    [baseline_metrics, tuned_metrics],
    keys=["Baseline", "Tuned"],
)

print("Baseline vs Tuned — hold-out validation:")
print(comparison)

tuned_metrics.to_csv("../metrics/random_forest_metrics.csv", index=False)
print("\nMetrics saved to ../metrics/random_forest_metrics.csv")

In [ ]:
pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_,
}).sort_values("Importance", ascending=False).head(15)

## 7. Prepare Submission

Retrain the tuned model on **all available training data** for the strongest
possible submission, then predict on the competition test set.

In [ ]:
final_rf = RandomForestRegressor(
    **best_params,
    random_state=42,
    n_jobs=-1,
)
final_rf.fit(X, y)
print(f"Final model trained on all {len(X):,} samples")

y_submission_pred = final_rf.predict(X_submission)
submission = pd.DataFrame({"id": test_ids, "exam_score": y_submission_pred})
submission.to_csv("../submission/random_forest_submission.csv", index=False)
print("Submission saved to ../submission/random_forest_submission.csv")
submission.head()

In [ ]:
joblib.dump(final_rf, "../models/random_forest.pkl")
print("Model saved to ../models/random_forest.pkl")